In [1]:
from deepagents import create_deep_agent
from langgraph.checkpoint.base import BaseCheckpointSaver
from langgraph.checkpoint.memory import InMemorySaver

from agentlab import middlewares
from agentlab.prompts import build_personal_assistant_prompt
from agentlab.tools import web_search_tool

"""Long-term memory wiring for the P5 assistant: virtual filesystem routes over a LangGraph store."""

from pathlib import Path

from deepagents.backends import (
    BackendProtocol,
    CompositeBackend,
    FilesystemBackend,
    StateBackend,
    StoreBackend,
)
from langgraph.runtime import Runtime

from agentlab.configs.personal_assistant_config import (
    MEMORIES_ROUTE_PREFIX,
    SKILLS_DIR,
    SKILLS_ROUTE_PREFIX,
)
from agentlab.schemas.context.user_context_schema import UserContext


MEMORIES_NAMESPACE_KIND = "memories"
SKILLS_DIR = Path(_) / "resources" / "skills"

def user_memory_ns(runtime: Runtime[UserContext] | None) -> tuple[str, str]:
    context = runtime.context if runtime is not None else None
    if not isinstance(context, UserContext) or not context.user_id.strip():
        raise ValueError("context must not be empty")
    return context.user_id, MEMORIES_NAMESPACE_KIND


def make_cbackend() -> CompositeBackend:
    return CompositeBackend(
        default=StateBackend(),
        routes={
            "/memories/":StoreBackend(
                namespace=user_memory_ns
            ),
            "/skills/":FilesystemBackend(
                root_dir=SKILLS_DIR,
                virtual_mode=True
            )

        }
    )

In [2]:
def make_personal_ag(store, backend: BackendProtocol|None = None, checkpointer: BaseCheckpointSaver|None = None):
    return create_deep_agent(
        model="openai: gpt-5.6-luna",
        system_prompt=build_personal_assistant_prompt(),
        tools=[web_search_tool.make_web_search_tool()],
        backend=backend if backend is not None else make_cbackend(),
        skills=["/skills/"],
        checkpointer=checkpointer if checkpointer is not None else InMemorySaver(),
        store=store,
        context_schema=UserContext,
        middleware=[middlewares.fallback_middleware.make_model_fallback()],
    )

In [ ]:
"""CLI for the P5 personal assistant: ``lab-assistant``.

Commands: ``/memory`` (show the active user's stored memories), ``/skills``
(list packaged skills), ``/new_thread`` (same user, fresh conversation),
``/user <id>`` (switch user; memories are isolated per user), ``/help``, ``quit``.
"""

import uuid
from pathlib import Path
from typing import Any

import yaml
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore

from agentlab.agents.personal_assistant_agent import make_personal_assistant_agent
from agentlab.configs.personal_assistant_config import (
    DEFAULT_ASSISTANT_PROVIDER,
    DEFAULT_ASSISTANT_THREAD_ID,
    DEFAULT_ASSISTANT_TIER,
    DEFAULT_USER_ID,
    SKILLS_DIR,
)
from agentlab.models import llm_model
from agentlab.state.context_state import UserContext
from agentlab.utils.console_util import configure_utf8_stdout
from agentlab.utils.get_text_util import get_text
from agentlab.utils.memory_backend_util import MEMORIES_NAMESPACE_KIND
from agentlab.utils.streaming_util import stream_agent_with_observability


_HELP = """Commands:
  /memory      show stored memories for the active user
  /skills      list available procedural skills
  /new_thread  start a new conversation thread for the same user
  /user <id>   switch the active user (starts a new thread)
  /help        show this help
  quit         end the session"""
_PREVIEW_LINES = 10




In [ ]:
def new_thread_id():
    return f"new_thread_{uuid.uuid4().hex[:8]}"

def _read_frontmatter(skill_file: Path):
    text = skill_file.read_text(encoding="utf-8")
    _, block, _ = text.split("---", 2)
    data = yaml.safe_load(block)
    return data if isinstance(data, dict) else {}

In [ ]:
def print_user_memories(store, user_id):
    items = store.search(
        (user_id, MEMORIES_NAMESPACE_KIND),
    )
    for item in items:
        value = item.values
        content = value.get